# 🌐 `get_domain_from_url` — Domain Extractor

> **Data Cleaning Tool** — Ekstrak `Domain Name` dan `TLD` dari daftar URL mentah.

| Step | Deskripsi |
|------|----------|
| 1️⃣  | Paste URL di cell input |
| 2️⃣  | Jalankan cell secara berurutan |
| 3️⃣  | Download hasil sebagai CSV |

---

Link to Google Colab: [here](https://colab.research.google.com/drive/1ZJAken_1N1caVM2800BNMPt8DquAsZbr?usp=sharing)

## 📦 Cell 1 — Install & Import Libraries

In [ ]:
# ── Install (jika belum tersedia) ──────────────────────────────────────────────
# tldextract adalah library terbaik untuk parsing domain + TLD secara akurat
!pip install tldextract --quiet

# ── Import ─────────────────────────────────────────────────────────────────────
import tldextract
import pandas as pd
from urllib.parse import urlparse
from datetime import datetime
import re
import time
from IPython.display import display, HTML

print("✅ Semua library berhasil diimport!")
print(f"   pandas     : v{pd.__version__}")
print(f"   tldextract : v{tldextract.__version__}")

## ✏️ Cell 2 — Input URL

Type or paste your URLs in the text field below (one per line), then click **Submit**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

text_area = widgets.Textarea(
    placeholder='Paste URLs here, one per line...\nhttps://www.example.com\nhttps://example.co.uk',
    layout=widgets.Layout(width='100%', height='200px'),
    style={'font_family': 'monospace'}
)

btn_submit = widgets.Button(
    description='Submit ▶',
    button_style='primary',
    layout=widgets.Layout(width='120px', height='36px')
)

out = widgets.Output()
url_list = []

def on_submit(b):
    global url_list
    url_list = [u.strip() for u in text_area.value.strip().splitlines() if u.strip()]
    with out:
        out.clear_output()
        print(f"📋 {len(url_list)} URL(s) loaded:")
        for i, u in enumerate(url_list, 1):
            print(f"   [{i:02d}] {u}")

btn_submit.on_click(on_submit)

display(widgets.Label('🔗 Enter URLs (one per line):'), text_area, btn_submit, out)

## ⚙️ Cell 3 — Fungsi Ekstraksi Domain

In [ ]:
def normalize_url(url: str) -> str:
    """Tambah scheme https:// jika belum ada."""
    url = url.strip()
    if not re.match(r'^https?://', url, re.IGNORECASE):
        url = 'https://' + url
    return url


def extract_domain_info(raw_url: str) -> dict:
    """
    Ekstrak domain name dan TLD dari sebuah URL.

    Returns dict dengan key:
        - domain_url   : URL asli (input)
        - domain_name  : domain lengkap tanpa www (e.g. jazzcars.co.uk)
        - domain_tld   : TLD / suffix (e.g. co.uk, com, net)
        - status       : 'ok' atau 'error'
        - note         : pesan error jika ada
    """
    try:
        url = normalize_url(raw_url)
        ext = tldextract.extract(url)

        if not ext.domain:
            return {
                'domain_url'  : raw_url,
                'domain_name' : None,
                'domain_tld'  : None,
                'status'      : 'error',
                'note'        : 'domain kosong / tidak valid'
            }

        # Susun domain_name = domain + suffix (tanpa www)
        domain_name = f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
        domain_tld  = ext.suffix or ext.domain

        return {
            'domain_url'  : raw_url,
            'domain_name' : domain_name,
            'domain_tld'  : domain_tld,
            'status'      : 'ok',
            'note'        : ''
        }

    except Exception as e:
        return {
            'domain_url'  : raw_url,
            'domain_name' : None,
            'domain_tld'  : None,
            'status'      : 'error',
            'note'        : str(e)
        }


print("✅ Fungsi ekstraksi siap!")
print()

# Quick test
test_cases = [
    "https://www.jazzcars.co.uk",
    "https://www.wealthywheels.com",
    "thebizbit.com",  # tanpa scheme
]
print("🧪 Quick test:")
for tc in test_cases:
    r = extract_domain_info(tc)
    print(f"   Input  : {tc}")
    print(f"   Domain : {r['domain_name']}   TLD: {r['domain_tld']}   [{r['status'].upper()}]")
    print()

## 🚀 Cell 4 — Jalankan Ekstraksi (dengan Log)

In [ ]:
# ── Logger sederhana ───────────────────────────────────────────────────────────
LOG_ICONS = {'INFO': '🔵', 'OK': '✅', 'WARN': '⚠️ ', 'ERROR': '❌', 'SYS': '⚡'}

def log(msg, level='INFO'):
    ts = datetime.now().strftime('%H:%M:%S.%f')[:12]
    icon = LOG_ICONS.get(level, '🔵')
    print(f"  {ts}  {icon} [{level:<5}]  {msg}")


# ── Jalankan ekstraksi ─────────────────────────────────────────────────────────
print("=" * 65)
print("  get_domain_from_url — Extraction Pipeline")
print("=" * 65)
print()

log("Pipeline started", 'SYS')
log(f"Input buffer: {len(url_list)} URL(s) loaded", 'INFO')
log("tldextract engine: aktif (database IANA)", 'INFO')
print()

records = []
ok_count  = 0
err_count = 0
start_time = time.time()

for i, url in enumerate(url_list, 1):
    result = extract_domain_info(url)
    records.append(result)

    if result['status'] == 'ok':
        ok_count += 1
        log(f"[{i:03d}] {result['domain_url']:<45} → {result['domain_name']}  (.{result['domain_tld']})", 'OK')
    else:
        err_count += 1
        log(f"[{i:03d}] {result['domain_url']:<45} → {result['note']}", 'ERROR')

elapsed = time.time() - start_time
print()
print("-" * 65)
log(f"Selesai dalam {elapsed:.3f}s", 'SYS')
log(f"Sukses : {ok_count}  |  Gagal : {err_count}  |  Total : {len(url_list)}", 'SYS')
print("=" * 65)

## 📊 Cell 5 — Tampilkan Hasil sebagai DataFrame

In [ ]:
# ── Buat DataFrame ─────────────────────────────────────────────────────────────
df = pd.DataFrame(records)[['domain_url', 'domain_name', 'domain_tld', 'status', 'note']]

# ── Summary stats ──────────────────────────────────────────────────────────────
df_ok  = df[df['status'] == 'ok']
df_err = df[df['status'] == 'error']
unique_tlds = df_ok['domain_tld'].nunique()

print(f"📊 Summary")
print(f"   Total URL  : {len(df)}")
print(f"   ✅ Sukses  : {len(df_ok)}")
print(f"   ❌ Error   : {len(df_err)}")
print(f"   🏷️  Unique TLD : {unique_tlds}")
print()

if unique_tlds > 0:
    tld_counts = df_ok['domain_tld'].value_counts()
    print("   Top TLDs:")
    for tld, cnt in tld_counts.items():
        bar = '█' * cnt
        print(f"   .{tld:<15} {bar} ({cnt})")
    print()

# ── Styling DataFrame untuk display ───────────────────────────────────────────
def style_status(val):
    if val == 'ok':    return 'background-color:#d4edda; color:#155724; font-weight:bold'
    if val == 'error': return 'background-color:#f8d7da; color:#721c24; font-weight:bold'
    return ''

display_df = df[['domain_url', 'domain_name', 'domain_tld', 'status']].copy()
display_df.index = display_df.index + 1  # mulai dari 1
display_df.index.name = '#'

styled = (
    display_df.style
    .applymap(style_status, subset=['status'])
    .set_properties(**{
        'font-family': 'monospace',
        'font-size': '13px'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#343a40'),
                  ('color','white'),
                  ('font-family','monospace'),
                  ('padding','8px 12px')]
    }])
)

display(styled)

## 💾 Cell 6 — Export & Download CSV

In [ ]:
from google.colab import files

# ── Siapkan DataFrame output (hanya kolom yang diperlukan) ─────────────────────
df_output = df[df['status'] == 'ok'][['domain_url', 'domain_name', 'domain_tld']].copy()
df_output.columns = ['domain URL', 'Domain Name', 'domain_tld']
df_output = df_output.reset_index(drop=True)

# ── Nama file dengan timestamp ─────────────────────────────────────────────────
timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
filename   = f"domains_{timestamp}.csv"

# ── Simpan ke file ─────────────────────────────────────────────────────────────
df_output.to_csv(filename, index=False, encoding='utf-8-sig')  # utf-8-sig agar Excel terbuka dengan benar

print(f"✅ File disimpan  : {filename}")
print(f"   Total baris    : {len(df_output)}")
print(f"   Kolom          : {list(df_output.columns)}")
print()
print("⬇️  Memulai download...")

# ── Auto-download di Google Colab ──────────────────────────────────────────────
files.download(filename)

## 🔄 (Opsional) Cell 7 — Batch dari File TXT / CSV

Kalau URL-nya banyak banget, bisa upload file `.txt` atau `.csv` langsung.

In [ ]:
from google.colab import files as colab_files
import io

print("📁 Upload file .txt (satu URL per baris) atau .csv (kolom bernama 'url')")
uploaded = colab_files.upload()

for fname, content in uploaded.items():
    print(f"\n🔍 Memproses file: {fname}")

    if fname.endswith('.csv'):
        df_in = pd.read_csv(io.BytesIO(content))
        # Cari kolom url secara case-insensitive
        url_col = next((c for c in df_in.columns if 'url' in c.lower()), df_in.columns[0])
        url_list_batch = df_in[url_col].dropna().tolist()
    else:
        url_list_batch = content.decode('utf-8').strip().splitlines()
        url_list_batch = [u.strip() for u in url_list_batch if u.strip()]

    print(f"   Ditemukan {len(url_list_batch)} URL")

    # Ekstrak
    batch_records = [extract_domain_info(u) for u in url_list_batch]
    df_batch = pd.DataFrame(batch_records)[['domain_url','domain_name','domain_tld','status']]
    df_batch.index = df_batch.index + 1

    ok_b  = (df_batch['status']=='ok').sum()
    err_b = (df_batch['status']=='error').sum()
    print(f"   ✅ Sukses: {ok_b}  ❌ Error: {err_b}")

    display(df_batch.head(20))

    # Export
    out_name = f"domains_batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_batch[df_batch['status']=='ok'][['domain_url','domain_name','domain_tld']].to_csv(out_name, index=False)
    colab_files.download(out_name)
    print(f"\n⬇️  Downloaded: {out_name}")

---
### 📌 Catatan

- Library **`tldextract`** menggunakan database TLD dari [Mozilla Public Suffix List](https://publicsuffix.org/) — akurat untuk multi-part TLD seperti `co.uk`, `com.au`, `co.id`, dll.
- URL tanpa `https://` akan otomatis dinormalisasi.
- Output CSV menggunakan encoding `utf-8-sig` agar bisa dibuka langsung di Excel tanpa masalah encoding.